<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">01 Data and Signals</h1>
<hr>


## **4. Converting Analog Signals to Digital Time Series**


<div class="alert alert-light">

  **Name:** (enter name here)
  
  **Matriculation Number:** (enter matriculation number here)
  
</div>

<style>
.reveal .jp-CodeCell .jp-Cell-inputWrapper,
.reveal .jp-CodeCell .jp-InputArea,
.reveal div.code_cell div.input,
.reveal div.code_cell .input_area {
    display: none !important;
}
</style>



### 4.1 Introduction

The previous notebook, [**Fourier Approximation of a Rectangular Pulse**](01.3_Fourier_Approximation_of_a_Rectangular_Pulse.ipynb), explored how frequency components represent a waveform and how rectangular pulses can represent sample values. We now examine how those samples are obtained and converted into digital data.

An analog signal can take values on a continuous scale, while digital data uses discrete symbols, often represented by binary `0`s and `1`s. An **analog-to-digital converter (ADC)** connects these representations. This notebook explains its three main operations: sampling, quantization, and encoding.


### 4.2 Analog-to-Digital Converter Pipeline

To convert a continuous-time signal into a bitstream, an ADC performs three main operations:

1. **Sampling:** Measure the signal at regular intervals $T_s$ to produce a discrete-time sequence $x[n]=x(nT_s)$.
2. **Quantization:** Round each sample $x[n]$ to the nearest value in a finite set of levels, yielding $x_Q[n]$.
3. **Encoding:** Map each quantized level to a fixed-length binary word, producing a bitstream with values $b_k\in\{0,1\}$.



The diagram illustrates this end-to-end pipeline. The following sections examine each step in detail.

<p align="center">
   <img src="../../.admin/notebooks/picture/ADC_Pipeline.png" alt="ADC Pipeline Diagram" width="100%"/>
</p>



### 4.3 Sampling

Sampling converts a continuous-time signal $x(t)$ into a discrete-time sequence $x[n]$ by taking measurements at regular intervals:

$$
x[n]=x(nT_s),
\quad n=0,1,2,\dots
$$

Here, $T_s$ is the **sampling interval**, measured in seconds. Sampling makes the time positions discrete; quantization, introduced later, makes the possible amplitude values discrete.


Each sample $x[n]$ records the value of the analog waveform at time $t_n=nT_s$. The **sampling rate** $f_s=1/T_s$ is the number of samples taken per second.

A higher sampling rate produces more samples over the same duration and allows higher-frequency components to be represented without aliasing. It also increases the amount of data. A lower sampling rate produces fewer samples, but may lose information about rapid changes in the signal.


#### 4.3.1 From Samples to a Zero-Order-Hold Signal

The sequence $x[n]$ contains values only at the discrete sampling instants $nT_s$; it does not specify a continuous waveform between those instants. A common reconstruction or hardware representation is the **zero-order hold (ZOH)**, which holds every sample value constant for one complete sampling interval. This produces a staircase-shaped signal that can be understood as a weighted row of rectangular pulses:

$$
x_{\mathrm{ZOH}}(t)=\sum_n x[n]\,\operatorname{rect}\!\left(\frac{t-(n+\tfrac12)T_s}{T_s}\right).
$$

Each rectangle has duration $T_s$, begins at $nT_s$, and has amplitude $x[n]$. The ZOH is one possible continuous-time representation of the sample sequence; it is not an inherent part of the abstract discrete-time data.

The preceding notebook [**Fourier Approximation of a Rectangular Pulse**](01.3_Fourier_Approximation_of_a_Rectangular_Pulse.ipynb) derives the spectrum of a rectangular pulse and explores its reconstruction from a finite set of frequency components.

Use the slider below to change how many samples are taken from a one-second sine wave. The stems mark the sample values; the smooth line shows the original waveform.


In [ ]:
from pathlib import Path
from runpy import run_path

_helper = next(root / ".admin/notebooks/analog_to_digital.py"
               for root in (Path.cwd(), *Path.cwd().parents)
               if (root / ".admin/notebooks/analog_to_digital.py").is_file())
_adc = run_path(str(_helper))
_adc["show_sampling"]()


#### 4.3.2 Nyquist-Shannon Sampling Theorem

The sampling rate $f_s$ must be chosen with the signal's frequency content in mind. Sampling too slowly can make a rapidly varying signal appear to have a different, lower frequency. Taking more samples helps only when they provide information that the lower rate could not capture; it does not remove noise or quantization errors by itself.

The plots below compare the same signal at sampling rates of 12 Hz and 100 Hz:

$$
x(t)=0.5\sin(2\pi 6t)+0.5\sin(2\pi 20t).
$$

The signal contains components at 6 Hz and 20 Hz. At 12 Hz, the red sample points do not uniquely identify the original waveform. At 100 Hz, both components can be represented without aliasing under the ideal sampling model.


In [ ]:
_adc["show_sampling_comparison"]()


The **Nyquist-Shannon sampling theorem** gives a sufficient sampling condition for a band-limited signal:

> If a continuous-time signal has no frequency components above $f_{\max}$, a sampling rate $f_s>2f_{\max}$ allows exact reconstruction under ideal conditions: exact, uniformly spaced samples over all time and ideal reconstruction.

Half the sampling rate, $f_s/2$, is called the **Nyquist frequency**. To avoid aliasing for the signals considered here, the highest signal frequency must lie below this limit. Sampling exactly at twice a sinusoid's frequency is not a safe general choice: depending on its phase, every sample could fall at a zero crossing.

As in notebook 01.3, the **Discrete Fourier Transform (DFT)** describes a finite sample block using frequency coefficients. For an even number $N$ of real-valued samples, the nonnegative-frequency bins are

$$
f_k=\frac{k}{N}f_s,
\quad k=0,1,\dots,\frac{N}{2}.
$$

They run from 0 to $f_s/2$. The negative-frequency coefficients contain the corresponding conjugate information, so we can display a one-sided spectrum. For odd $N$, the last nonnegative-frequency bin lies just below $f_s/2$.


In the example above, the highest component is $f_{\max}=20\,\mathrm{Hz}$, so we choose

$$
f_s>2f_{\max}=40\,\mathrm{Hz}.
$$

The 100 Hz sampling rate meets this condition; 12 Hz does not. The same reasoning explains why a 44.1 kHz audio sampling rate leaves a margin above twice a 20 kHz signal bandwidth.


#### 4.3.3 Aliasing

Sampling cannot distinguish every possible continuous-time waveform. When the sampling rate is too low for the signal's bandwidth, different frequencies can produce the same samples. This ambiguity is called **aliasing**: a high-frequency component can appear as a lower-frequency component in the sampled data.

For example, a 20 Hz sinusoid sampled at 30 Hz appears at 10 Hz in the sampled spectrum, since $|20-30|=10$. Its apparent phase may also change. The samples alone cannot tell us which of these frequencies was present in the original signal. Filtering out frequencies above the usable band before sampling prevents this ambiguity.

A practical **anti-aliasing filter** attenuates unwanted high-frequency components before the ADC takes samples. It cannot change abruptly from passing wanted frequencies to rejecting unwanted ones: it needs a transition band. Choosing a sampling rate above twice the highest wanted frequency leaves room for this transition below the Nyquist frequency. A larger margin makes filtering easier, but produces more samples to store or transmit.

The interactive plot below shows the samples, their one-sided amplitude spectrum, and a waveform reconstructed using sinc interpolation. Change the sampling rate and compare the spectral peaks with the original 6 Hz and 20 Hz components. The reconstruction uses only the one-second sample block shown, so finite-record effects can cause differences, particularly near its ends, even when the sampling rate is high enough to avoid aliasing.


In [ ]:
_adc["show_aliasing"]()


A familiar example of aliasing is the **wagon-wheel effect**: a rotating wheel can appear to turn slowly, stop, or rotate backwards in a film. A camera records the wheel only at the times of its frames. If the wheel has identical spokes, several physical orientations look the same, making its motion ambiguous between frames. The apparent motion depends on the frame rate, the wheel's speed, and the number of identical spokes.


### 4.4 Quantization

Sampling makes the time positions discrete; **quantization** makes the possible amplitude values discrete. On a graph, sampling selects positions along the horizontal time axis, while quantization selects levels along the vertical amplitude axis.

In these examples, a quantizer rounds each input to one of $L=2^B$ equally spaced levels, where $B$ is the number of bits per sample. The levels include both endpoints of the range $[x_{\min},x_{\max}]$, so their spacing is

$$
\Delta=\frac{x_{\max}-x_{\min}}{L-1}.
$$

Here, $\Delta$ is the **quantization step size**. The plots use $x_{\min}=0$ and $x_{\max}=1$.


#### 4.4.1 Uniform Quantization

For the range $[0,1]$ used here, a **uniform nearest-level quantizer** maps an input value $x[n]$ to

$$
Q(x[n])=\Delta\cdot\operatorname{round}\!\left(\frac{x[n]}{\Delta}\right),
\qquad \Delta=\frac{1}{2^B-1}.
$$

The function $\operatorname{round}$ selects the nearest integer. Dividing by $\Delta$ expresses the input in units of one quantization step; rounding selects a level, and multiplying by $\Delta$ converts it back to an amplitude.

Use the bit-count slider below to compare the original sine and ramp signals with their quantized versions. More bits provide more levels and smaller steps, giving finer amplitude resolution. Fewer bits give coarser resolution but require fewer bits to store each sample.


In [ ]:
_adc["show_quantization"]()


#### 4.4.2 Quantization Error

The **quantization error** is the difference between a quantized value and its original input:

$$
e[n]=Q(x[n])-x[n].
$$

For nearest-level rounding, an input within the quantizer's range is at most half a step away from its nearest level. Thus,

$$
-\frac{\Delta}{2}\le e[n]\le\frac{\Delta}{2}.
$$

This bound assumes that the input stays within the representable range; it does not cover overload or clipping. Increasing the number of bits reduces $\Delta$ and therefore reduces the maximum rounding error.

The dashed lines in the error plot mark $\pm\Delta/2$. Its vertical scale changes with the selected bit count, so compare the numerical axis values as well as the shape of the curve.


In [ ]:
_adc["show_quantization_error"]()


### 4.5 Encoding

Once a sample has been assigned to one of $L$ quantization levels, its level index must be represented by a binary word for storage or transmission. With $B$ bits per sample, $L=2^B$ levels can be represented. Each level index $i\in\{0,\dots,L-1\}$ is mapped to its $B$-bit binary representation.

This **fixed-length encoding** uses the same number of bits for every level, regardless of how often that level occurs.

For one signal channel sampled at $f_s$ samples per second and encoded with $B$ bits per sample, the **raw data rate** is

$$
R=f_sB \quad \text{bits per second}.
$$

This counts only the sample data, without compression, packet headers, or error-control coding. Increasing $f_s$ gives more samples per second; increasing $B$ gives finer amplitude resolution. Both increase the data rate, but they address different limitations: more bits cannot undo aliasing, and more samples do not reduce the quantization step size.

#### 4.5.1 Example: 3-bit Encoding

The example below uses a sine wave quantized to eight levels. Each quantized sample is labelled with the three-bit word representing its level index.


In [ ]:
_adc["show_encoding"]()


---

### 4.6 Hands-On Exercise

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>



**A signal contains frequencies up to 3 kHz. Using the sampling condition in Section 4.3.2, what rate must the sampling frequency exceed to avoid aliasing? Give one suitable sampling rate.**

(answer here)


**In the aliasing example in Section 4.3.3, set the sampling rate to 30 Hz. Identify the two peaks in the sampled spectrum. Which peak corresponds to the original 20 Hz component, and which frequency replaces that component in the reconstructed waveform? Explain why the samples alone cannot distinguish the original component from its lower-frequency alias.**

(answer here)


**Use the bit-count slider in Section 4.4.1 to compare one-bit and eight-bit quantization over $[0,1]$. For each setting, calculate the number of levels $L$ and the step size $\Delta$. Describe one visible difference between the quantized waveforms and explain it using your calculated values.**

(answer here)


**For the three-bit quantizer over the range $[0,1]$, calculate the step size $\Delta$ and the maximum absolute rounding error. Set the slider in Section 4.4.2 to three bits and check your error bound against the dashed lines.**

(answer here)


**A sample has value $x[n]=0.6$. Using the three-bit nearest-level quantizer over $[0,1]$, calculate its level index $i$, its quantized value $Q(x[n])$, and its signed error $e[n]=Q(x[n])-x[n]$. Write the three-bit binary word that encodes the level index, including any leading zeros. Show your calculation.**

(answer here)


**OPTIONAL: Design an ADC configuration for one sensor channel with wanted frequencies up to 3 kHz and amplitude values within $[0,1]$. The available sampling rates are 6, 8, and 12 kHz, and the bit depth can be any integer from 1 to 8. Use the sampling condition in Section 4.3.2 and assume that an appropriate anti-aliasing filter removes unwanted out-of-band components. The maximum absolute rounding error must be no greater than 0.01. Choose the lowest available sampling rate and the smallest bit depth that meet these requirements. Show how you checked both conditions, then calculate the raw data rate using $R=f_sB$, in bit/s and kbit/s (1 kbit/s = 1000 bit/s). Explain why increasing the bit depth cannot compensate for an insufficient sampling rate.**

**Hints**

1. **Sampling rate:** Find $f_{\max}$ and apply $f_s>2f_{\max}$. The inequality is strict. Check the available rates and choose the smallest one that satisfies it.
2. **Bit depth:** For the endpoint-inclusive quantizer over $[0,1]$, use $\Delta=1/(2^B-1)$ and the error bound $|e[n]|\le\Delta/2$. You therefore need $1/[2(2^B-1)]\le0.01$. Try integer values of $B$ and check that the next smaller value fails the requirement.
3. **Data rate:** Convert your chosen sampling rate from kHz to samples per second before multiplying by $B$. Divide the result in bit/s by 1000 to obtain kbit/s.
4. **Explain the trade-off:** Sampling rate controls how often the signal is measured; bit depth controls how finely each measured amplitude is represented. If two different waveforms produce the same samples, would storing those sample values more precisely tell you which waveform was present?

(answer here)


**OPTIONAL: Were any explanations or tasks unclear? Identify any missing information or sections that could be improved.**

(Please help the teaching team by describing what was unclear.)
